# Loan Amortization Payoff Scenario

This cleaned notebook converts the original loan calculator into a reproducible payoff scenario model. It compares a baseline fixed-rate loan against an accelerated payoff plan with extra principal applied early.

The model is intentionally dependency-light: it uses only the Python standard library so the amortization assumptions remain easy to inspect.

## Model assumptions

- Fixed-rate, fully amortizing loan.
- Interest accrues monthly on the opening balance.
- The scheduled payment is applied first.
- Extra principal is applied after the scheduled payment on the same payment date.
- Extra payments are capped at the remaining principal balance.

This is a scenario model for analysis and education, not financial advice.

In [1]:
from datetime import date

from loan_amortization import (
    LoanScenario,
    build_schedule,
    compare_scenarios,
    currency,
    default_scenarios,
    monthly_payment,
    print_summary,
    summarize,
)

In [2]:
LOAN_AMOUNT = 359_200.00
INTEREST_RATE = 0.0625
TERM_MONTHS = 30 * 12
START_DATE = date(2023, 11, 1)

EXTRA_PAYMENTS = {
    date(2023, 11, 1): 4_000.00,
    date(2023, 12, 1): 4_000.00,
    date(2024, 1, 1): 5_000.00,
}

base_monthly_payment = monthly_payment(LOAN_AMOUNT, INTEREST_RATE, TERM_MONTHS)
print(currency(base_monthly_payment))

$2,211.66


## Build the baseline and accelerated schedules

The baseline uses only the required monthly payment. The accelerated scenario adds `$13,000` of principal during the first three payments.

In [3]:
baseline = LoanScenario(
    name='Baseline',
    principal=LOAN_AMOUNT,
    annual_rate=INTEREST_RATE,
    term_months=TERM_MONTHS,
    start_date=START_DATE,
)

accelerated = LoanScenario(
    name='Accelerated payoff',
    principal=LOAN_AMOUNT,
    annual_rate=INTEREST_RATE,
    term_months=TERM_MONTHS,
    start_date=START_DATE,
    extra_payments=EXTRA_PAYMENTS,
)

print_summary([baseline, accelerated])

Scenario                  Payment   Months       Payoff        Interest      Total paid        Extra
----------------------------------------------------------------------------------------------------
Baseline                $2,211.66      360   2053-10-01     $436,993.48     $796,193.48        $0.00
Accelerated payoff      $2,211.66      326   2050-12-01     $373,606.21     $732,806.21   $13,000.00


In [4]:
impact = compare_scenarios(baseline, accelerated)

print(f"Months saved: {impact['months_saved']} ({impact['years_saved']:.2f} years)")
print(f"Interest saved: {currency(impact['interest_saved'])}")
print(f"Extra principal applied: {currency(impact['total_extra_principal'])}")
print(
    "Interest saved per extra principal dollar: "
    f"${impact['interest_saved_per_extra_dollar']:.2f}"
)
print(f"Baseline payoff date: {impact['baseline_payoff_date'].isoformat()}")
print(f"Accelerated payoff date: {impact['alternative_payoff_date'].isoformat()}")

Months saved: 34 (2.83 years)
Interest saved: $63,387.27
Extra principal applied: $13,000.00
Interest saved per extra principal dollar: $4.88
Baseline payoff date: 2053-10-01
Accelerated payoff date: 2050-12-01


## Inspect the early-payment mechanics

The first three rows show the extra principal being applied immediately after the scheduled payment. The lower balance then reduces interest accrual for every later month.

In [5]:
accelerated_schedule = build_schedule(accelerated)

header = (
    f"{'Date':<12} {'Payment':>11} {'Interest':>11} {'Principal':>12} "
    f"{'Extra':>10} {'Balance':>13}"
)
print(header)
print('-' * len(header))
for row in accelerated_schedule[:6]:
    print(
        f"{row['date'].isoformat():<12} "
        f"{currency(row['scheduled_payment']):>11} "
        f"{currency(row['interest']):>11} "
        f"{currency(row['scheduled_principal']):>12} "
        f"{currency(row['extra_principal']):>10} "
        f"{currency(row['ending_balance']):>13}"
    )

Date             Payment    Interest    Principal      Extra       Balance
--------------------------------------------------------------------------
2023-11-01     $2,211.66   $1,870.83      $340.83  $4,000.00   $354,859.17
2023-12-01     $2,211.66   $1,848.22      $363.44  $4,000.00   $350,495.73
2024-01-01     $2,211.66   $1,825.50      $386.16  $5,000.00   $345,109.57
2024-02-01     $2,211.66   $1,797.45      $414.21      $0.00   $344,695.36
2024-03-01     $2,211.66   $1,795.29      $416.37      $0.00   $344,278.99
2024-04-01     $2,211.66   $1,793.12      $418.54      $0.00   $343,860.45


## Takeaway

In this scenario, the `$13,000` early principal reduction cuts the payoff timeline from `360` payments to `326`, avoids `$63,387.27` in interest, and moves the payoff date from `2053-10-01` to `2050-12-01`.

The useful pattern is not the exact loan amount. It is the repeatable comparison: define the amortization assumptions, create scenario schedules, and evaluate the delta in payoff date and lifetime interest.